## Explores the results of the linked table created by link_dams_stations.slurm

In [1]:
import pandas as pd

In [2]:
link_table = pd.read_csv("../data/processed/dam_station_links_19020401.csv")
link_table.head()

,huc8,dam_id,dam_name,station_id,dam_reach_id,year_removed,river,station_reach_id,relationship,distance_to_dam_km,network_distance_km,distance_method,dam_snap_distance_m,station_snap_distance_m,dam_snap_fraction,station_snap_fraction
0,19020401,AK-010,Lower Eklutna River Dam,USGS-15272970,75000100004714,2017,Eklutna,75000100002520,unconnected,NaN,NaN,NaN,227.764842,17.116171,0.521747,0.302300
1,19020401,AK-010,Lower Eklutna River Dam,USGS-15273020,75000100004714,2017,Eklutna,75000100000099,unconnected,NaN,NaN,NaN,227.764842,7.968422,0.521747,0.977877
2,19020401,AK-010,Lower Eklutna River Dam,USGS-15273030,75000100004714,2017,Eklutna,75000100001891,unconnected,NaN,NaN,NaN,227.764842,13.624231,0.521747,0.515974
3,19020401,AK-010,Lower Eklutna River Dam,USGS-15273040,75000100004714,2017,Eklutna,75000100000059,unconnected,NaN,NaN,NaN,227.764842,27.595394,0.521747,0.138479
4,19020401,AK-010,Lower Eklutna River Dam,USGS-15273045,75000100004714,2017,Eklutna,75000100000059,unconnected,NaN,NaN,NaN,227.764842,3.819037,0.521747,0.464142


In [3]:
import geopandas as gpd
import folium
import branca.colormap as cm
import numpy as np

# Load snapped geometries.
snapped_path = "../data/processed/snapped_features_19020401.gpkg"

stations = gpd.read_file(snapped_path, layer="stations_snapped")
dams = gpd.read_file(snapped_path, layer="dams_snapped")

# Inspect identifiers if needed.
print("Link table:", link_table.columns.tolist())
print("\nStation identifiers:", stations.columns.tolist())
print("\nDam identifiers:", dams.columns.tolist())

Link table: ['huc8', 'dam_id', 'dam_name', 'station_id', 'dam_reach_id', 'year_removed', 'river', 'station_reach_id', 'relationship', 'distance_to_dam_km', 'network_distance_km', 'distance_method', 'dam_snap_distance_m', 'station_snap_distance_m', 'dam_snap_fraction', 'station_snap_fraction']

Station identifiers: ['OrganizationIdentifier', 'OrganizationFormalName', 'MonitoringLocationIdentifier', 'MonitoringLocationName', 'MonitoringLocationTypeName', 'MonitoringLocationDescriptionText', 'HUCEightDigitCode', 'DrainageAreaMeasure/MeasureValue', 'DrainageAreaMeasure/MeasureUnitCode', 'ContributingDrainageAreaMeasure/MeasureValue', 'ContributingDrainageAreaMeasure/MeasureUnitCode', 'LatitudeMeasure', 'LongitudeMeasure', 'SourceMapScaleNumeric', 'HorizontalAccuracyMeasure/MeasureValue', 'HorizontalAccuracyMeasure/MeasureUnitCode', 'HorizontalCollectionMethodName', 'HorizontalCoordinateReferenceSystemDatumName', 'VerticalMeasure/MeasureValue', 'VerticalMeasure/MeasureUnitCode', 'VerticalAc

In [4]:
# Select the first dam in the link table.
selected_dam_row = link_table.iloc[0]

dam_id = selected_dam_row["dam_id"]
dam_name = selected_dam_row["dam_name"]
year_removed = selected_dam_row["year_removed"]

print("Selected dam:", repr(dam_id), repr(dam_name), repr(year_removed))

print("\nMatching dam by unique AR_ID:")
print(
    dams.loc[
        dams["AR_ID"].astype(str) == str(dam_id),
        ["AR_ID", "Dam_Name", "Year_Removed", "NHDPlusID"]
    ].to_string(index=False)
)

print("\nData types:")
print("Link table year:", link_table["year_removed"].dtype)
print("Dam layer year:", dams["Year_Removed"].dtype)

Selected dam: 'AK-010' 'Lower Eklutna River Dam' np.int64(2017)

Matching dam by unique AR_ID:
 AR_ID                Dam_Name Year_Removed    NHDPlusID
AK-010 Lower Eklutna River Dam         2017 7.500010e+13

Data types:
Link table year: int64
Dam layer year: str


In [5]:
from pathlib import Path

import geopandas as gpd
import folium
import branca.colormap as cm
import networkx as nx
import numpy as np
import pandas as pd

# --------------------------------------------------
# 1. Select a dam removal and its linked stations
# --------------------------------------------------

selected_dam_id = link_table["dam_id"].iloc[0]

links = link_table[
    link_table["dam_id"].astype(str) == str(selected_dam_id)
].copy()

selected_dam = dams[
    dams["AR_ID"].astype(str) == str(selected_dam_id)
]

if selected_dam.empty:
    raise ValueError(f"No dam found for AR_ID {selected_dam_id}.")

dam = selected_dam.iloc[0]
dam_point = dam.geometry
dam_name = dam["Dam_Name"]
year_removed = dam["Year_Removed"]

# Keep only stations with a connected network relationship and distance.
links = links[
    links["relationship"].isin(
        ["upstream", "downstream", "same_reach"]
    )
    & links["distance_to_dam_km"].notna()
].copy()

# Match link records to the snapped station geometries.
stations["station_id"] = (
    stations["MonitoringLocationIdentifier"].astype(str)
)
links["station_id"] = links["station_id"].astype(str)

mapped_stations = stations.merge(
    links,
    on="station_id",
    how="inner",
    suffixes=("", "_link"),
)

mapped_stations = mapped_stations[
    mapped_stations.geometry.notna()
    & ~mapped_stations.geometry.is_empty
].copy()

# Signed distance: upstream negative, downstream positive.
def signed_distance(row):
    distance = float(row["distance_to_dam_km"])

    if row["relationship"] == "upstream":
        return -distance
    if row["relationship"] == "downstream":
        return distance
    return 0.0  # same reach

mapped_stations["signed_distance_km"] = mapped_stations.apply(
    signed_distance, axis=1
)

# --------------------------------------------------
# 2. Load NHDPlus stream reaches and flow connections
# --------------------------------------------------

huc8 = str(links["huc8"].iloc[0])

gdb_path = (
    Path("../data/processed/nhdplus_extracted")
    / f"NHDPLUS_H_{huc8}_HU8_GDB"
    / f"NHDPLUS_H_{huc8}_HU8_GDB.gdb"
)

if not gdb_path.exists():
    raise FileNotFoundError(f"NHDPlus geodatabase not found: {gdb_path}")

streams = gpd.read_file(gdb_path, layer="NHDFlowline")
flow = gpd.read_file(gdb_path, layer="NHDPlusFlow")

# Normalize reach identifiers.
streams["NHDPlusID"] = pd.to_numeric(
    streams["NHDPlusID"], errors="coerce"
)
streams = streams.dropna(subset=["NHDPlusID"]).copy()
streams["NHDPlusID"] = streams["NHDPlusID"].astype("int64")
streams = streams.drop_duplicates("NHDPlusID")

length_by_reach = (
    streams.set_index("NHDPlusID")["LengthKM"].to_dict()
)

for col in ["FromNHDPID", "ToNHDPID"]:
    flow[col] = pd.to_numeric(flow[col], errors="coerce")

stream_ids = set(length_by_reach)

valid_flow = flow[
    flow["FromNHDPID"].isin(stream_ids)
    & flow["ToNHDPID"].isin(stream_ids)
    & flow["FromNHDPID"].notna()
    & flow["ToNHDPID"].notna()
    & flow["FromNHDPID"].ne(0)
    & flow["ToNHDPID"].ne(0)
].copy()

valid_flow["FromNHDPID"] = valid_flow["FromNHDPID"].astype("int64")
valid_flow["ToNHDPID"] = valid_flow["ToNHDPID"].astype("int64")

# Assumption: FromNHDPID -> ToNHDPID follows downstream flow.
G = nx.DiGraph()
G.add_nodes_from(stream_ids)
G.add_edges_from(
    zip(valid_flow["FromNHDPID"], valid_flow["ToNHDPID"])
)

# Use source-reach lengths to choose shortest network paths.
for u, v in G.edges:
    G[u][v]["weight"] = float(length_by_reach.get(u, 0) or 0)

dam_reach = int(float(dam["NHDPlusID"]))
dam_fraction = float(dam["snap_fraction"])
dam_length = float(length_by_reach[dam_reach])

# --------------------------------------------------
# 3. Find the stream paths connecting the dam and stations
# --------------------------------------------------

# Signed distance to the approximate midpoint of each displayed reach.
reach_distances = {dam_reach: 0.0}

for _, station in mapped_stations.iterrows():
    station_reach = pd.to_numeric(
        pd.Series([station["NHDPlusID"]]), errors="coerce"
    ).iloc[0]

    if pd.isna(station_reach):
        continue

    station_reach = int(station_reach)

    if station_reach not in G or dam_reach not in G:
        continue

    relationship = station["relationship"]

    if relationship == "same_reach":
        path = [dam_reach]

    elif relationship == "upstream":
        # Follow flow downstream from the station's reach to the dam.
        try:
            path = nx.shortest_path(
                G,
                source=station_reach,
                target=dam_reach,
                weight="weight",
            )
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

    else:
        # Follow flow downstream from the dam to the station's reach.
        try:
            path = nx.shortest_path(
                G,
                source=dam_reach,
                target=station_reach,
                weight="weight",
            )
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            continue

    # Estimate distance to each reach midpoint along this path.
    for i, reach in enumerate(path):
        if reach == dam_reach:
            distance = 0.0

        elif relationship == "upstream":
            intermediate = sum(
                float(length_by_reach.get(r, 0) or 0)
                for r in path[i + 1:-1]
            )
            distance = -(
                dam_fraction * dam_length
                + intermediate
                + 0.5 * float(length_by_reach.get(reach, 0) or 0)
            )

        elif relationship == "downstream":
            intermediate = sum(
                float(length_by_reach.get(r, 0) or 0)
                for r in path[1:i]
            )
            distance = (
                (1 - dam_fraction) * dam_length
                + intermediate
                + 0.5 * float(length_by_reach.get(reach, 0) or 0)
            )

        else:
            distance = 0.0

        # If a reach appears in multiple paths, retain the distance
        # with the smallest absolute magnitude.
        if (
            reach not in reach_distances
            or abs(distance) < abs(reach_distances[reach])
        ):
            reach_distances[reach] = distance

# Keep only the stream geometries that occur on these paths.
reach_gdf = streams[
    streams["NHDPlusID"].isin(reach_distances)
].copy()

reach_gdf["signed_distance_km"] = reach_gdf["NHDPlusID"].map(
    reach_distances
)

reach_gdf = reach_gdf.to_crs("EPSG:4326")
mapped_stations = mapped_stations.to_crs("EPSG:4326")
dams = dams.to_crs("EPSG:4326")
dam_point = dams.loc[
    dams["AR_ID"].astype(str) == str(selected_dam_id),
    "geometry",
].iloc[0]

print(f"Dam: {dam_name} ({year_removed})")
print(f"Connected stations: {len(mapped_stations):,}")
print(f"Stream reaches on station paths: {len(reach_gdf):,}")

# --------------------------------------------------
# 4. Create the map and distance color ramp
# --------------------------------------------------

max_distance = max(
    float(reach_gdf["signed_distance_km"].abs().max())
    if not reach_gdf.empty else 1.0,
    float(mapped_stations["signed_distance_km"].abs().max())
    if not mapped_stations.empty else 1.0,
    1.0,
)

colors = cm.LinearColormap(
    colors=["#08306B", "#006D77", "#542788", "#B35806", "#B2182B"],
    vmin=-max_distance,
    vmax=max_distance,
    caption=(
        "Estimated network distance (km) - upstream is negative, downstream is positive "
    ),
)

m = folium.Map(
    location=[dam_point.y, dam_point.x],
    zoom_start=11,
    tiles="OpenStreetMap",
)

# Draw stream reaches, colored by estimated distance.
# Convert datetime columns to strings for GeoJSON serialization.
reach_gdf_map = reach_gdf.copy()

for col in reach_gdf_map.columns:
    if pd.api.types.is_datetime64_any_dtype(reach_gdf_map[col]):
        reach_gdf_map[col] = reach_gdf_map[col].astype(str)

folium.GeoJson(
    reach_gdf_map.to_json(),
    name="Connected stream reaches",
    style_function=lambda feature: {
        "color": colors(
            feature["properties"]["signed_distance_km"]
        ),
        "weight": 3,
        "opacity": 0.85,
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["NHDPlusID", "signed_distance_km"],
        aliases=["Reach ID:", "Estimated distance (km):"],
        localize=True,
    ),
).add_to(m)

# Draw water-quality stations.
for _, station in mapped_stations.iterrows():
    distance = station["signed_distance_km"]

    color = (
        "#7b3294"
        if station["relationship"] == "same_reach"
        else colors(distance)
    )

    folium.CircleMarker(
        location=[station.geometry.y, station.geometry.x],
        radius=6,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.95,
        weight=1,
        tooltip=(
            f"{station['MonitoringLocationName']}<br>"
            f"{station['relationship']}<br>"
            f"{distance:.2f} km from dam"
        ),
    ).add_to(m)

# Mark the dam removal.
folium.Marker(
    location=[dam_point.y, dam_point.x],
    tooltip=f"{dam_name} ({year_removed})",
    popup=f"{dam_name}<br>Removal year: {year_removed}<br>AR_ID: {selected_dam_id}",
    icon=folium.Icon(color="black", icon="flag"),
).add_to(m)

colors.add_to(m)
folium.LayerControl().add_to(m)

# Fit the map to the displayed features.
bounds = [[dam_point.y, dam_point.x]]

for geom in reach_gdf.geometry:
    if geom is not None and not geom.is_empty:
        minx, miny, maxx, maxy = geom.bounds
        bounds.extend([[miny, minx], [maxy, maxx]])

for geom in mapped_stations.geometry:
    if geom is not None and not geom.is_empty:
        bounds.append([geom.y, geom.x])

m.fit_bounds(bounds)

m

/home/creighton.jo/.conda/envs/dam-removal/lib/python3.11/site-packages/pyogrio/raw.py:200: UserWarning: Measured (M) geometry types are not supported. Original type 'Measured 3D MultiLineString' is converted to 'MultiLineString Z'
  return ogr_read(


Dam: Lower Eklutna River Dam (2017)
Connected stations: 26
Stream reaches on station paths: 88


### Save these subsets for later analysis

In [6]:
from pathlib import Path
import geopandas as gpd

output_path = Path("../data/processed/catchment_map_layers.gpkg")
output_path.unlink(missing_ok=True)

# Stream reaches already selected for the existing map
reach_gdf.to_file(
    output_path,
    layer="streams",
    driver="GPKG",
)

# Water-quality stations linked to the selected dam
mapped_stations.to_file(
    output_path,
    layer="stations",
    driver="GPKG",
)

# Selected dam removal
dam_gdf = gpd.GeoDataFrame(
    [{
        "dam_name": dam_name,
        "year_removed": year_removed,
    }],
    geometry=[dam_point],
    crs=dams.crs,
)

dam_gdf.to_file(
    output_path,
    layer="dam",
    driver="GPKG",
)

print(f"Saved map layers to {output_path}")

Saved map layers to ../data/processed/catchment_map_layers.gpkg


In [7]:
# Export the mapped station IDs for the WQP downloader.
station_ids_path = Path(
    "../data/raw/wqp/results/pilot_station_selection.csv"
)

station_ids_path.parent.mkdir(parents=True, exist_ok=True)

mapped_stations[
    ["MonitoringLocationIdentifier"]
].drop_duplicates().to_csv(station_ids_path, index=False)

print(
    f"Saved {mapped_stations['MonitoringLocationIdentifier'].nunique()} "
    f"unique station IDs to {station_ids_path}"
)

Saved 26 unique station IDs to ../data/raw/wqp/results/pilot_station_selection.csv
